In [1]:
import pysam
import os

In [2]:
def analyze_bam(bam_filename):
    """
    Analyze a paired-end BAM file with the following filters:
      - Process only read1 records (so each read pair is only counted once).
      - Read must not be secondary or supplementary.
      - The read's reference name must equal its mate's reference name
        (used as a proxy for concordant pairing; the BAM "properly paired"
        flag itself is not checked).
      - Mapping quality must be >= 20.

    For each read passing the filters, the UMI is extracted as the first 15 bases
    of the query sequence (read.query_sequence)
    The function tallies:
      - Total read1 count per reference.
      - Unique 15bp UMIs observed per reference.

    Output is written as a TSV file in the "counts" directory. The TSV filename is
    derived from the BAM file's basename by replacing .bam with .MAPQ20.tsv.

    Returns:
      dict: A dictionary keyed by reference name; each value is a dict with keys:
            "read_count" and "umi_count".
    """
    bam = pysam.AlignmentFile(bam_filename, "rb")
    counts = {}  # { ref_name: {"read_count": int, "umi_set": set() } }
    
    for read in bam.fetch(until_eof=True):
        # Only process read1 to count each pair only once.
        if not read.is_read1:
            continue
        
        if read.is_secondary or read.is_supplementary:
            continue
        
        # Ensure that the mate maps to the same reference.
        if read.reference_id != read.next_reference_id:
            continue
            
        # Extract the reference name from the SAM record (RNAME field).
        # This yields the full string as present in the alignment (e.g. "EH38D3316369_D253X").
        ref_name = bam.get_reference_name(read.reference_id)
        
        # Filter by mapping quality.
        if read.mapping_quality < 20:
            continue 
        
        # Extract the UMI: first 15 bases of the query sequence.
        seq = read.query_sequence
        if seq is None or len(seq) < 15:
            continue
        umi = seq[:15]
        
        # Initialize the dictionary entry if needed.
        if ref_name not in counts:
            counts[ref_name] = {"read_count": 0, "umi_set": set()}
        
        counts[ref_name]["read_count"] += 1
        counts[ref_name]["umi_set"].add(umi)
    
    bam.close()
    
    # Build final results: count the number of unique UMIs for each reference.
    results = {}
    for ref_name, data in counts.items():
        results[ref_name] = {
            "read_count": data["read_count"],
            "umi_count": len(data["umi_set"])
        }
    
    # Prepare output path. The output file is named the same as the BAM file, 
    # but with a .tsv extension, written to a subdirectory called "counts".
    bam_basename = os.path.basename(bam_filename)
    out_filename = bam_basename.replace(".bam", ".MAPQ20.tsv")
    counts_dir = "counts"
    os.makedirs(counts_dir, exist_ok=True)
    out_path = os.path.join(counts_dir, out_filename)
    
    # Write the results to the TSV file.
    with open(out_path, "w") as fout:
        fout.write("Reference\tRead_Count\tUMI_Count\n")
        for ref, data in results.items():
            fout.write(f"{ref}\t{data['read_count']}\t{data['umi_count']}\n")
    
    print(f"Output written to {out_path}")
    return results

In [3]:
# Path to the file-of-file-names (fofn) that contains the BAM file prefixes.
prefixes_file = "<path_to_prefixes_fofn>"

# Open and iterate through each line in the prefixes file.
with open(prefixes_file, "r") as infile:
    for line in infile:
        # Remove any trailing newline or whitespace.
        prefix = line.strip()
        if not prefix:
            continue  # Skip blank lines.

        # Construct the full BAM filename. Here, the files are in ./250410_alignments/ and have a .bam suffix.
        bam_filename = os.path.join("<path_to_bam_directory>", prefix + ".bam")

        # Print a message indicating which file is being processed.
        print(f"Processing file: {bam_filename}")

        # Call the analyze_bam function on the current BAM file.
        # analyze_bam is assumed to be defined (as shown in the previous function).
        analyze_bam(bam_filename)